# 04 · Whisper (VAD) + clasificación LLM (E.6 paso 7)

Ejecutar con **Kernel limpio → Run All**, en el orden documentado en `docs/NOTEBOOKS.md`.

Si hay modelos provisionados (`scripts/fetch_models.py`) se usan faster-whisper y llama.cpp; si no, este notebook usa los backends simulados de `tests/fakes.py` y lo indica.

In [ ]:
import sys; sys.path.insert(0, '.')
from _common import *
import os, json
from podcleany.config import Config
from podcleany.models import make_backends
from podcleany import audio, decision
W = workdir(); cfg = Config.load(W/'data')
stt, clf, warns = make_backends(cfg)
print(stt.name, clf.name, warns)
if stt.name == 'none':
    sys.path.insert(0, str(ROOT)); os.environ['PODCLEANY_FAKE_TRUTH'] = str(W/'truth.json')
    (W/'truth.json').write_text(json.dumps([[0,8,'strong'],[39.5,59.5,'strong'],[92.5,107.5,'weak']]))
    from tests.fakes import make_backends as fake; stt, clf, _ = fake(cfg); print('usando backends simulados')
x = audio.read_range(W/'blocks', 0, 134, 16000, 60)
utts = stt.transcribe(x, 0.0)
cands = decision.llm_candidates(decision.group_utterances(utts), clf)
print([(c['start'], c['end'], max(c['ps'])) for c in cands])
json.dump(cands, open(W/'cands.json','w'))